# todo
1. scene/gaussian_model.py
    - restore(given)
2. gaussian_renderer/__init__.py (given)
3. cuda code
    - forward.cu

# arguments/__init__.py


In [ ]:
class PipelineParams(ParamGroup):
    def __init__(self, parser):
        self.convert_SHs_python = False
        self.compute_cov3D_python = False
        self.debug = False
        self.env_map_res = 0
        self.env_optimize_until = 1000000000
        self.env_optimize_from = 0
        self.eval_shfs_4d = False

        # ========= added line start here =========
        self.opa_threshold = 0.05
        # ========= added line end here =========
        
        super().__init__(parser, "Pipeline Parameters")

# gaussian_renderer/diff_gaussian_rasterization.py

In [ ]:
@staticmethod
def forward(
     ctx,
     means3D,
     means2D,
     sh,
     colors_precomp,
     flow_2d,
     opacities,
     ts,
     scales,
     scales_t,
     rotations,
     rotations_r,
     cov3Ds_precomp,
     raster_settings,
):
     args = (
          raster_settings.bg,
          means3D,
          colors_precomp,
          flow_2d,
          opacities,
          ts,
          scales,
          scales_t,
          rotations,
          rotations_r,
          raster_settings.scale_modifier,
          cov3Ds_precomp,
          # ────────── static Gaussian parameters (forward) ──────────
          raster_settings.static_xyz,
          raster_settings.static_features_dc,
          raster_settings.static_features_rest,
          raster_settings.static_scaling,
          raster_settings.static_rotation,
          raster_settings.static_opacity,
          raster_settings.static_max_radii2D,
          raster_settings.static_denom,
          raster_settings.static_xyz_gradient_accum,
          # ────────────────────────────────────────────────────────────
          raster_settings.viewmatrix,
          raster_settings.projmatrix,
          raster_settings.tanfovx,
          raster_settings.tanfovy,
          raster_settings.image_height,
          raster_settings.image_width,
          sh,
          raster_settings.sh_degree,
          raster_settings.sh_degree_t,
          raster_settings.campos,
          raster_settings.timestamp,
          raster_settings.time_duration,
          raster_settings.rot_4d,
          raster_settings.gaussian_dim,
          raster_settings.force_sh_3d,
          raster_settings.prefiltered,
          raster_settings.debug,
     )
#    …
@staticmethod
def backward(ctx, grad_out_color, grad_radii, grad_depth, grad_alpha, grad_flow, grad_covs_com):
     raster_settings = ctx.raster_settings
#    …
     args = (
          raster_settings.bg,
          means3D,
          out_means3D,
          radii,
          colors_precomp,
          flow_2d,
          opacities,
          ts,
          scales,
          scales_t,
          rotations,
          rotations_r,
          raster_settings.scale_modifier,
          cov3Ds_precomp,
          # ────────── static Gaussian parameters (backward) ─────────
          raster_settings.static_xyz,
          raster_settings.static_features_dc,
          raster_settings.static_features_rest,
          raster_settings.static_scaling,
          raster_settings.static_rotation,
          raster_settings.static_opacity,
          raster_settings.static_max_radii2D,
          raster_settings.static_denom,
          raster_settings.static_xyz_gradient_accum,
          # ────────────────────────────────────────────────────────────
          raster_settings.viewmatrix,
          raster_settings.projmatrix,
          raster_settings.tanfovx,
          raster_settings.tanfovy,
          grad_out_color,
          grad_depth,
          grad_alpha,
          grad_flow,
          sh,
          raster_settings.sh_degree,
          raster_settings.sh_degree_t,
          raster_settings.campos,
          raster_settings.timestamp,
          raster_settings.time_duration,
          raster_settings.rot_4d,
          raster_settings.gaussian_dim,
          raster_settings.force_sh_3d,
          geomBuffer,
          num_rendered,
          binningBuffer,
          imgBuffer,
          raster_settings.debug
     )
#    …


# scene/gaussian_model.py

In [ ]:
def __init__(self, sh_degree: int, gaussian_dim: int = 3, …):
    …
    self.setup_functions()

    # --- Hybrid 3D–4D용 static Gaussian placeholder -------------
    # restore() 시 unpack 될 필드들이 미리 있어야 합니다.
    self.static_xyz                = torch.empty(0)              # [N_static × 3]
    self.static_features_dc        = torch.empty(0)              # [N_static × DC]
    self.static_features_rest      = torch.empty(0)              # [N_static × Rest]
    self.static_scaling            = torch.empty(0)              # [N_static × 3]
    self.static_rotation           = torch.empty(0)              # [N_static × 4]
    self.static_opacity            = torch.empty(0)              # [N_static × 1]
    self.static_max_radii2D        = torch.empty(0, dtype=torch.int64)
    self.static_denom              = torch.empty(0)              # [N_static × 1]
    self.static_xyz_gradient_accum = torch.empty(0)              # [N_static × 1]

def capture(self):
    if self.gaussian_dim == 3:
        return ( … )       # (기존 3D 리턴 그대로)
    elif self.gaussian_dim == 4:
        return (
            self.active_sh_degree,
            self._xyz,
            self._features_dc,
            self._features_rest,
            self._scaling,
            self._rotation,
            self._opacity,
            self.max_radii2D,
            self.xyz_gradient_accum,
            self.t_gradient_accum,
            self.denom,
            self.optimizer.state_dict(),
            self.spatial_lr_scale,
            self._t,
            self._scaling_t,
            self._rotation_r,
            self.rot_4d,
            self.env_map,
            self.active_sh_degree_t,
            # ────────── 여기부터 새로운 static 필드 ──────────
            self.static_xyz,
            self.static_features_dc,
            self.static_features_rest,
            self.static_scaling,
            self.static_rotation,
            self.static_opacity,
            self.static_max_radii2D,
            self.static_denom,
            self.static_xyz_gradient_accum,
        )